In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML"
)

ML_FILE = (
    PROJECT_ROOT
    / "data"
    / "ml_ready"
    / "thermal_context_ml_ready_event_level.csv"
)

data = pd.read_csv(ML_FILE)

print("Dataset shape:", data.shape)
print("\nColumns:")
print(data.columns.tolist())

Dataset shape: (439251, 33)

Columns:
['acq_date', 'lat_cell', 'lon_cell', 'event_number', 'start_time', 'end_time', 'observation_count', 'latitude', 'longitude', 'mean_frp', 'peak_frp', 'total_frp', 'mean_brightness', 'persistent', 'historical_detection_count', 'historical_mean_frp', 'local_frp_deviation', 'historical_daily_activity', 'previously_detected', 'duration_minutes', 'frp_range', 'frp_peak_ratio', 'frp_per_observation', 'start_hour', 'day_of_week', 'month', 'is_weekend', 'long_duration', 'is_day', 'is_night', 'anomaly_score', 'anomaly_prediction', 'is_anomaly']


In [2]:
event_key = [
    "acq_date",
    "start_time",
    "end_time",
    "latitude",
    "longitude"
]

print("Total rows:", len(data))

print(
    "Unique events:",
    data[event_key].drop_duplicates().shape[0]
)

print(
    "Duplicate events:",
    data.duplicated(
        subset=event_key
    ).sum()
)

Total rows: 439251
Unique events: 439251
Duplicate events: 0


In [3]:
missing = (
    data.isna()
    .sum()
    .sort_values(ascending=False)
)

missing = missing[missing > 0]

print("Columns with missing values:")
print(missing)

Columns with missing values:
frp_peak_ratio         393
duration_minutes       382
month                  382
is_anomaly             382
is_weekend             382
is_day                 382
long_duration          382
anomaly_score          382
anomaly_prediction     382
frp_range              382
frp_per_observation    382
day_of_week            382
start_hour             382
is_night               382
dtype: int64


In [4]:
# infinite vslues 
numeric_cols = data.select_dtypes(
    include=np.number
).columns

inf_counts = np.isinf(
    data[numeric_cols]
).sum()

print(
    "Total infinite values:",
    inf_counts.sum()
)

print(
    inf_counts[inf_counts > 0]
)

Total infinite values: 0
Series([], dtype: int64)


In [5]:
numeric_summary = data[
    numeric_cols
].describe().T

display(numeric_summary)

,count,mean,std,min,25%,50%,75%,max
lat_cell,439251.0,2304.238281,453.793489,8.960000e+02,2033.000000,2340.00000,2592.000000,3828.000000
lon_cell,439251.0,8161.547917,575.425421,6.850000e+03,7743.000000,8071.00000,8439.000000,9698.000000
event_number,439251.0,1.149648,0.423349,1.000000e+00,1.000000,1.00000,1.000000,8.000000
observation_count,439251.0,1.600379,1.281113,1.000000e+00,1.000000,1.00000,2.000000,26.000000
latitude,439251.0,20.763406,4.088233,8.078340e+00,18.321790,21.08345,23.355195,34.494130
longitude,439251.0,81.620466,5.754242,6.850890e+01,77.434970,80.71685,84.392800,96.986540
mean_frp,439251.0,6.914934,17.005489,0.000000e+00,2.030000,3.80000,6.653333,2056.400000
peak_frp,439251.0,8.372936,24.993534,0.000000e+00,2.160000,4.00000,7.280000,2056.400000
total_frp,439251.0,13.641015,60.299692,0.000000e+00,2.480000,4.70000,9.730000,4853.380000
mean_brightness,439251.0,329.852963,16.198321,2.073800e+02,314.700000,334.78000,340.960000,495.100000


In [6]:
checks = {
    "observation_count": (0, np.inf),
    "duration_minutes": (0, np.inf),
    "mean_frp": (0, np.inf),
    "peak_frp": (0, np.inf),
    "total_frp": (0, np.inf),
    "frp_range": (0, np.inf),
    "frp_peak_ratio": (0, np.inf),
    "frp_per_observation": (0, np.inf),
    "mean_brightness": (0, np.inf),
    "historical_detection_count": (0, np.inf),
    "historical_daily_activity": (0, np.inf)
}

for col, (low, high) in checks.items():
    if col in data.columns:
        bad = (
            (data[col] < low) |
            (data[col] > high)
        ).sum()

        print(f"{col}: {bad} suspicious rows")

observation_count: 0 suspicious rows
duration_minutes: 0 suspicious rows
mean_frp: 0 suspicious rows
peak_frp: 0 suspicious rows
total_frp: 0 suspicious rows
frp_range: 27 suspicious rows
frp_peak_ratio: 0 suspicious rows
frp_per_observation: 0 suspicious rows
mean_brightness: 0 suspicious rows
historical_detection_count: 0 suspicious rows
historical_daily_activity: 0 suspicious rows


In [7]:
data["acq_date"] = pd.to_datetime(
    data["acq_date"],
    errors="coerce"
)

data["start_time"] = pd.to_datetime(
    data["start_time"],
    errors="coerce"
)

data["end_time"] = pd.to_datetime(
    data["end_time"],
    errors="coerce"
)

print("Date range:")
print("Start:", data["start_time"].min())
print("End:  ", data["end_time"].max())

Date range:
Start: 2025-01-01 03:50:00
End:   2025-03-19 21:31:00


In [8]:
bad_time = (
    data["end_time"] < data["start_time"]
).sum()

print(
    "Events where end_time < start_time:",
    bad_time
)

Events where end_time < start_time: 0


In [9]:
data["calculated_duration"] = (
    data["end_time"] -
    data["start_time"]
).dt.total_seconds() / 60

duration_difference = (
    data["calculated_duration"] -
    data["duration_minutes"]
).abs()

print(
    "Duration inconsistencies:",
    (duration_difference > 0.01).sum()
)

Duration inconsistencies: 0


In [10]:
print(
    data["is_anomaly"]
    .value_counts()
    .sort_index()
)

print("\nAnomaly percentage:")

print(
    data["is_anomaly"].mean() * 100,
    "%"
)

is_anomaly
0.0    356687
1.0     82182
Name: count, dtype: int64

Anomaly percentage:
18.725861247889462 %


In [11]:
print(
    data["anomaly_score"].describe()
)

count    438869.000000
mean          0.052018
std           0.070082
min          -0.312441
25%           0.017846
50%           0.068020
75%           0.106416
max           0.136635
Name: anomaly_score, dtype: float64


In [12]:
leakage_features = [
    "historical_detection_count",
    "historical_mean_frp",
    "local_frp_deviation",
    "historical_daily_activity",
    "previously_detected"
]

display(
    data[
        leakage_features
    ].describe().T
)

,count,mean,std,min,25%,50%,75%,max
historical_detection_count,439251.0,14.328468,63.146672,0.00,0.00000,1.00,3.000,1571.000000
historical_mean_frp,439251.0,3.848882,12.490944,0.00,0.00000,1.35,3.765,1121.475833
local_frp_deviation,439251.0,3.049971,13.657870,-680.01,0.31879,1.90,4.100,2056.400000
historical_daily_activity,439251.0,13.134106,62.190388,0.00,0.00000,0.00,1.000,1551.000000
previously_detected,439251.0,0.605508,0.488742,0.00,0.00000,1.00,1.000,1.000000


In [13]:
print(
    "Historical detection count = 0:",
    (
        data["historical_detection_count"] == 0
    ).mean() * 100,
    "%"
)

Historical detection count = 0: 39.44919874968981 %


In [14]:
print(
    data[
        [
            "observation_count",
            "mean_frp",
            "peak_frp",
            "total_frp",
            "historical_detection_count",
            "historical_mean_frp",
            "local_frp_deviation",
            "historical_daily_activity"
        ]
    ].corr()["peak_frp"].sort_values(
        ascending=False
    )
)

peak_frp                      1.000000
mean_frp                      0.896065
total_frp                     0.825322
historical_mean_frp           0.609843
local_frp_deviation           0.419014
observation_count             0.244954
historical_detection_count   -0.019858
historical_daily_activity    -0.027150
Name: peak_frp, dtype: float64


In [15]:
report = {
    "total_rows": len(data),
    "unique_events": data[event_key]
        .drop_duplicates()
        .shape[0],

    "duplicate_events": data
        .duplicated(subset=event_key)
        .sum(),

    "missing_values": int(
        data.isna().sum().sum()
    ),

    "infinite_values": int(
        np.isinf(
            data.select_dtypes(
                include=np.number
            )
        ).sum().sum()
    ),

    "bad_time_events": int(
        (data["end_time"] < data["start_time"]).sum()
    ),

    "date_start": str(
        data["acq_date"].min()
    ),

    "date_end": str(
        data["acq_date"].max()
    )
}

print("========== DATA QUALITY REPORT ==========")

for key, value in report.items():
    print(f"{key}: {value}")

========== DATA QUALITY REPORT ==========
total_rows: 439251
unique_events: 439251
duplicate_events: 0
missing_values: 5359
infinite_values: 0
bad_time_events: 0
date_start: 2025-01-01 00:00:00
date_end: 2025-03-19 00:00:00


In [16]:
# ============================================================
# MISSING VALUE ANALYSIS
# ============================================================

missing_report = (
    data.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_report = missing_report[
    missing_report > 0
]

print("Columns containing missing values:")
print(missing_report)

print("\nMissing percentage:")
print(
    (missing_report / len(data) * 100)
    .round(4)
)

Columns containing missing values:
frp_peak_ratio         393
is_weekend             382
frp_per_observation    382
frp_range              382
anomaly_prediction     382
month                  382
is_night               382
is_day                 382
long_duration          382
is_anomaly             382
duration_minutes       382
anomaly_score          382
start_hour             382
day_of_week            382
dtype: int64

Missing percentage:
frp_peak_ratio         0.0895
is_weekend             0.0870
frp_per_observation    0.0870
frp_range              0.0870
anomaly_prediction     0.0870
month                  0.0870
is_night               0.0870
is_day                 0.0870
long_duration          0.0870
is_anomaly             0.0870
duration_minutes       0.0870
anomaly_score          0.0870
start_hour             0.0870
day_of_week            0.0870
dtype: float64


In [17]:
# ============================================================
# SHOW ROWS WITH MISSING VALUES
# ============================================================

missing_cols = missing_report.index.tolist()

display(
    data[
        data[missing_cols].isna().any(axis=1)
    ][
        event_key + missing_cols
    ].head(10)
)

,acq_date,start_time,end_time,latitude,longitude,frp_peak_ratio,is_weekend,frp_per_observation,frp_range,anomaly_prediction,month,is_night,is_day,long_duration,is_anomaly,duration_minutes,anomaly_score,start_hour,day_of_week
871,2025-01-01,2025-01-01 07:16:00,2025-01-01 07:41:00,21.057533,81.165493,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
889,2025-01-01,2025-01-01 19:46:00,2025-01-01 20:55:00,21.078748,85.035886,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1777,2025-01-01,2025-01-01 07:16:00,2025-01-01 08:04:00,22.042246,83.734661,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1845,2025-01-01,2025-01-01 07:16:00,2025-01-01 08:04:00,22.122003,83.476057,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3189,2025-01-02,2025-01-02 19:25:00,2025-01-02 20:14:00,20.806923,85.256619,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4786,2025-01-02,2025-01-02 19:25:00,2025-01-02 19:51:00,22.369753,87.303030,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5015,2025-01-02,2025-01-02 19:25:00,2025-01-02 20:13:00,23.674463,86.086627,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5115,2025-01-02,2025-01-02 19:25:00,2025-01-02 20:13:00,23.780933,86.373380,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6378,2025-01-03,2025-01-03 19:08:00,2025-01-03 19:55:00,20.787266,85.264079,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6617,2025-01-03,2025-01-03 19:08:00,2025-01-03 19:55:00,21.078098,85.035876,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# Check the 382 events with missing anomaly information

anomaly_missing = data[data["anomaly_score"].isna()].copy()

print("Events missing anomaly information:", len(anomaly_missing))

display(
    anomaly_missing[
        [
            "acq_date",
            "start_time",
            "end_time",
            "latitude",
            "longitude",
            "observation_count",
            "mean_frp",
            "peak_frp"
        ]
    ].head(20)
)

Events missing anomaly information: 382


,acq_date,start_time,end_time,latitude,longitude,observation_count,mean_frp,peak_frp
871,2025-01-01,2025-01-01 07:16:00,2025-01-01 07:41:00,21.057533,81.165493,3,7.170000,11.93
889,2025-01-01,2025-01-01 19:46:00,2025-01-01 20:55:00,21.078748,85.035886,9,2.254444,6.60
1777,2025-01-01,2025-01-01 07:16:00,2025-01-01 08:04:00,22.042246,83.734661,7,12.842857,20.95
1845,2025-01-01,2025-01-01 07:16:00,2025-01-01 08:04:00,22.122003,83.476057,3,5.410000,8.48
3189,2025-01-02,2025-01-02 19:25:00,2025-01-02 20:14:00,20.806923,85.256619,7,1.750000,2.26
4786,2025-01-02,2025-01-02 19:25:00,2025-01-02 19:51:00,22.369753,87.303030,3,0.996667,1.07
5015,2025-01-02,2025-01-02 19:25:00,2025-01-02 20:13:00,23.674463,86.086627,3,1.450000,1.95
5115,2025-01-02,2025-01-02 19:25:00,2025-01-02 20:13:00,23.780933,86.373380,7,1.130000,1.81
6378,2025-01-03,2025-01-03 19:08:00,2025-01-03 19:55:00,20.787266,85.264079,9,1.697778,2.50
6617,2025-01-03,2025-01-03 19:08:00,2025-01-03 19:55:00,21.078098,85.035876,9,1.933333,3.20


In [19]:
# Check whether these events have unusual values

print("Missing anomaly events statistics:")
print(anomaly_missing[
    [
        "observation_count",
        "mean_frp",
        "peak_frp"
    ]
].describe())

Missing anomaly events statistics:
       observation_count    mean_frp    peak_frp
count         382.000000  382.000000  382.000000
mean            5.840314   13.184384   29.769738
std             3.256836   25.844455   65.578666
min             3.000000    0.596667    0.780000
25%             3.000000    1.863929    2.700000
50%             6.000000    4.198333    6.205000
75%             7.000000    9.387619   17.280000
max            23.000000  220.054545  500.000000


In [20]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML")

events = pd.read_csv(
    PROJECT_ROOT / "data/events/thermal_events_prototype.csv"
)

anomaly = pd.read_csv(
    PROJECT_ROOT / "data/results/thermal_anomaly_prototype.csv"
)

data = pd.read_csv(
    PROJECT_ROOT / "data/ml_ready/thermal_context_ml_ready_event_level.csv"
)

# Convert date/time columns
for df in [events, anomaly, data]:
    df["acq_date"] = pd.to_datetime(df["acq_date"], errors="coerce")

print("Thermal events:", len(events))
print("Anomaly results:", len(anomaly))
print("Final ML dataset:", len(data))

Thermal events: 439251
Anomaly results: 439251
Final ML dataset: 439251


In [21]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML")

events = pd.read_csv(
    PROJECT_ROOT / "data/events/thermal_events_prototype.csv"
)

anomaly = pd.read_csv(
    PROJECT_ROOT / "data/results/thermal_anomaly_prototype.csv"
)

data = pd.read_csv(
    PROJECT_ROOT / "data/ml_ready/thermal_context_ml_ready_event_level.csv"
)

# Convert date/time columns
for df in [events, anomaly, data]:
    df["acq_date"] = pd.to_datetime(df["acq_date"], errors="coerce")

print("Thermal events:", len(events))
print("Anomaly results:", len(anomaly))
print("Final ML dataset:", len(data))

Thermal events: 439251
Anomaly results: 439251
Final ML dataset: 439251


In [22]:
missing_keys = events.merge(
    anomaly[event_key],
    on=event_key,
    how="left",
    indicator=True
)

missing_keys = missing_keys[
    missing_keys["_merge"] == "left_only"
]

print("Thermal events without anomaly result:", len(missing_keys))

display(
    missing_keys[
        event_key + [
            "observation_count",
            "mean_frp",
            "peak_frp"
        ]
    ].head(20)
)

Thermal events without anomaly result: 382


,acq_date,start_time,end_time,latitude,longitude,observation_count,mean_frp,peak_frp
871,2025-01-01,2025-01-01 07:16:00,2025-01-01 07:41:00,21.057533,81.165493,3,7.170000,11.93
889,2025-01-01,2025-01-01 19:46:00,2025-01-01 20:55:00,21.078748,85.035886,9,2.254444,6.60
1777,2025-01-01,2025-01-01 07:16:00,2025-01-01 08:04:00,22.042246,83.734661,7,12.842857,20.95
1845,2025-01-01,2025-01-01 07:16:00,2025-01-01 08:04:00,22.122003,83.476057,3,5.410000,8.48
3189,2025-01-02,2025-01-02 19:25:00,2025-01-02 20:14:00,20.806923,85.256619,7,1.750000,2.26
4786,2025-01-02,2025-01-02 19:25:00,2025-01-02 19:51:00,22.369753,87.303030,3,0.996667,1.07
5015,2025-01-02,2025-01-02 19:25:00,2025-01-02 20:13:00,23.674463,86.086627,3,1.450000,1.95
5115,2025-01-02,2025-01-02 19:25:00,2025-01-02 20:13:00,23.780933,86.373380,7,1.130000,1.81
6378,2025-01-03,2025-01-03 19:08:00,2025-01-03 19:55:00,20.787266,85.264079,9,1.697778,2.50
6617,2025-01-03,2025-01-03 19:08:00,2025-01-03 19:55:00,21.078098,85.035876,9,1.933333,3.20


In [23]:
# Check whether the missing events have unusual values

print("Missing anomaly events:", len(missing_keys))

print("\nObservation count:")
print(missing_keys["observation_count"].value_counts().sort_index().head(20))

print("\nFRP statistics:")
display(
    missing_keys[
        ["observation_count", "mean_frp", "peak_frp"]
    ].describe()
)

Missing anomaly events: 382

Observation count:
observation_count
3     177
6      22
7     108
9      43
11     14
12      1
13      5
14      6
15      1
17      1
18      2
19      1
23      1
Name: count, dtype: int64

FRP statistics:


,observation_count,mean_frp,peak_frp
count,382.000000,382.000000,382.000000
mean,5.840314,13.184384,29.769738
std,3.256836,25.844455,65.578666
min,3.000000,0.596667,0.780000
25%,3.000000,1.863929,2.700000
50%,6.000000,4.198333,6.205000
75%,7.000000,9.387619,17.280000
max,23.000000,220.054545,500.000000


In [24]:
# Check for zero/negative values that could break feature calculations

print("peak_frp <= 0:",
      (missing_keys["peak_frp"] <= 0).sum())

print("mean_frp <= 0:",
      (missing_keys["mean_frp"] <= 0).sum())

print("observation_count <= 0:",
      (missing_keys["observation_count"] <= 0).sum())

peak_frp <= 0: 0
mean_frp <= 0: 0
observation_count <= 0: 0


In [25]:
feature_file = PROJECT_ROOT / "data/features/thermal_features_prototype.csv"

thermal_features = pd.read_csv(feature_file)

print("Thermal feature rows:", len(thermal_features))

print("\nMissing values in thermal features:")
print(
    thermal_features.isna().sum()
    .sort_values(ascending=False)
    .head(20)
)

Thermal feature rows: 439251

Missing values in thermal features:
acq_date               0
start_time             0
end_time               0
latitude               0
longitude              0
observation_count      0
duration_minutes       0
mean_frp               0
peak_frp               0
total_frp              0
frp_range              0
frp_peak_ratio         0
frp_per_observation    0
mean_brightness        0
start_hour             0
day_of_week            0
month                  0
is_weekend             0
persistent             0
long_duration          0
dtype: int64


In [26]:
# Convert date/time columns to the same datatype

thermal_features["acq_date"] = pd.to_datetime(
    thermal_features["acq_date"],
    errors="coerce"
)

thermal_features["start_time"] = pd.to_datetime(
    thermal_features["start_time"],
    errors="coerce"
)

thermal_features["end_time"] = pd.to_datetime(
    thermal_features["end_time"],
    errors="coerce"
)

events["acq_date"] = pd.to_datetime(
    events["acq_date"],
    errors="coerce"
)

events["start_time"] = pd.to_datetime(
    events["start_time"],
    errors="coerce"
)

events["end_time"] = pd.to_datetime(
    events["end_time"],
    errors="coerce"
)

print("Date/time types fixed.")

Date/time types fixed.


In [27]:
check_features = events.merge(
    thermal_features,
    on=[
        "acq_date",
        "start_time",
        "end_time",
        "latitude",
        "longitude"
    ],
    how="left",
    indicator=True
)

missing_feature_events = check_features[
    check_features["_merge"] == "left_only"
]

print(
    "Events missing from thermal_features:",
    len(missing_feature_events)
)

Events missing from thermal_features: 382


In [28]:
# ============================================================
# STANDARDIZE ALL EVENT KEY DATATYPES
# ============================================================

event_key = [
    "acq_date",
    "start_time",
    "end_time",
    "latitude",
    "longitude"
]

# Date/time columns
datetime_cols = [
    "acq_date",
    "start_time",
    "end_time"
]

for df_name, df in {
    "events": events,
    "anomaly": anomaly,
    "thermal_features": thermal_features,
    "missing_keys": missing_keys
}.items():

    for col in datetime_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")

    # Make coordinates numeric
    df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

print("All event-key datatypes standardized.")

print("\nevents:")
print(events[event_key].dtypes)

print("\nthermal_features:")
print(thermal_features[event_key].dtypes)

print("\nmissing_keys:")
print(missing_keys[event_key].dtypes)

All event-key datatypes standardized.

events:
acq_date      datetime64[us]
start_time    datetime64[us]
end_time      datetime64[us]
latitude             float64
longitude            float64
dtype: object

thermal_features:
acq_date      datetime64[us]
start_time    datetime64[us]
end_time      datetime64[us]
latitude             float64
longitude            float64
dtype: object

missing_keys:
acq_date      datetime64[us]
start_time    datetime64[us]
end_time      datetime64[us]
latitude             float64
longitude            float64
dtype: object


In [29]:
# Remove the old merge indicator if it exists
if "_merge" in missing_keys.columns:
    missing_keys = missing_keys.drop(columns="_merge")

print("Columns cleaned.")

Columns cleaned.


In [30]:
missing_anomaly_check = missing_keys.merge(
    thermal_features,
    on=event_key,
    how="left",
    indicator="feature_match"
)

missing_from_features = (
    missing_anomaly_check["feature_match"] == "left_only"
).sum()

print(
    "Missing anomaly events:", len(missing_keys)
)

print(
    "Missing from thermal_features:",
    missing_from_features
)

Missing anomaly events: 382
Missing from thermal_features: 382


In [31]:
if missing_from_features > 0:
    print("\nThese events are missing from thermal_features:")
    display(
        missing_anomaly_check[
            missing_anomaly_check["feature_match"] == "left_only"
        ][event_key].head(20)
    )
else:
    print(
        "\nAll missing anomaly events are present in thermal_features."
    )


These events are missing from thermal_features:


,acq_date,start_time,end_time,latitude,longitude
0,2025-01-01,2025-01-01 07:16:00,2025-01-01 07:41:00,21.057533,81.165493
1,2025-01-01,2025-01-01 19:46:00,2025-01-01 20:55:00,21.078748,85.035886
2,2025-01-01,2025-01-01 07:16:00,2025-01-01 08:04:00,22.042246,83.734661
3,2025-01-01,2025-01-01 07:16:00,2025-01-01 08:04:00,22.122003,83.476057
4,2025-01-02,2025-01-02 19:25:00,2025-01-02 20:14:00,20.806923,85.256619
5,2025-01-02,2025-01-02 19:25:00,2025-01-02 19:51:00,22.369753,87.303030
6,2025-01-02,2025-01-02 19:25:00,2025-01-02 20:13:00,23.674463,86.086627
7,2025-01-02,2025-01-02 19:25:00,2025-01-02 20:13:00,23.780933,86.373380
8,2025-01-03,2025-01-03 19:08:00,2025-01-03 19:55:00,20.787266,85.264079
9,2025-01-03,2025-01-03 19:08:00,2025-01-03 19:55:00,21.078098,85.035876


In [32]:
# Check missing values in the anomaly dataset itself

print("Anomaly rows:", len(anomaly))

print("\nMissing values in anomaly dataset:")
print(
    anomaly.isna()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

Anomaly rows: 439251

Missing values in anomaly dataset:
frp_peak_ratio         11
start_time              0
end_time                0
latitude                0
longitude               0
observation_count       0
duration_minutes        0
mean_frp                0
acq_date                0
peak_frp                0
total_frp               0
frp_range               0
frp_per_observation     0
mean_brightness         0
start_hour              0
day_of_week             0
month                   0
is_weekend              0
persistent              0
long_duration           0
dtype: int64


In [33]:
# Check whether anomaly dataset contains duplicate event keys

print(
    "Duplicate anomaly event keys:",
    anomaly.duplicated(subset=event_key).sum()
)

Duplicate anomaly event keys: 0


In [34]:
# Check whether the anomaly dataset is missing exactly these 382 events

anomaly_check = events.merge(
    anomaly[event_key],
    on=event_key,
    how="left",
    indicator="anomaly_match"
)

missing_anomaly_events = anomaly_check[
    anomaly_check["anomaly_match"] == "left_only"
]

print(
    "Events missing from anomaly dataset:",
    len(missing_anomaly_events)
)

Events missing from anomaly dataset: 382


In [35]:
# Get the thermal features corresponding to the 382 missing anomaly events

missing_event_features = thermal_features.merge(
    missing_keys[event_key],
    on=event_key,
    how="inner"
)

print("Matched missing-anomaly events:", len(missing_event_features))

print("\nColumns:")
print(missing_event_features.columns.tolist())

Matched missing-anomaly events: 0

Columns:
['acq_date', 'start_time', 'end_time', 'latitude', 'longitude', 'observation_count', 'duration_minutes', 'mean_frp', 'peak_frp', 'total_frp', 'frp_range', 'frp_peak_ratio', 'frp_per_observation', 'mean_brightness', 'start_hour', 'day_of_week', 'month', 'is_weekend', 'persistent', 'long_duration', 'is_day', 'is_night']


In [36]:
feature_columns = [
    "duration_minutes",
    "frp_range",
    "frp_peak_ratio",
    "frp_per_observation",
    "start_hour",
    "day_of_week",
    "month",
    "is_weekend",
    "is_day",
    "is_night",
    "long_duration"
]

print("Missing values in the 382 events:\n")

print(
    missing_event_features[feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

Missing values in the 382 events:

duration_minutes       0
frp_range              0
frp_peak_ratio         0
frp_per_observation    0
start_hour             0
day_of_week            0
month                  0
is_weekend             0
is_day                 0
is_night               0
long_duration          0
dtype: int64


In [37]:
import numpy as np

numeric_features = missing_event_features[
    feature_columns
].select_dtypes(include=np.number)

print("Infinite values:\n")

print(
    np.isinf(numeric_features)
    .sum()
    .sort_values(ascending=False)
)

Infinite values:

duration_minutes       0
frp_range              0
frp_peak_ratio         0
frp_per_observation    0
start_hour             0
day_of_week            0
month                  0
is_weekend             0
is_day                 0
is_night               0
long_duration          0
dtype: int64


In [38]:
display(
    missing_event_features[
        event_key + feature_columns
    ].head(20)
)

,acq_date,start_time,end_time,latitude,longitude,duration_minutes,frp_range,frp_peak_ratio,frp_per_observation,start_hour,day_of_week,month,is_weekend,is_day,is_night,long_duration


In [39]:
# Recreate the exact ML feature list used for Isolation Forest

ml_features = [
    "observation_count",
    "duration_minutes",
    "mean_frp",
    "peak_frp",
    "total_frp",
    "frp_range",
    "frp_peak_ratio",
    "frp_per_observation",
    "mean_brightness",
    "start_hour",
    "day_of_week",
    "month",
    "is_weekend",
    "persistent",
    "long_duration",
    "is_day",
    "is_night"
]

X_check = thermal_features[ml_features].copy()

print("Original thermal feature rows:", len(X_check))

print("\nMissing values:")
print(X_check.isna().sum().sort_values(ascending=False))

print("\nInfinite values:")
print(
    np.isinf(
        X_check.select_dtypes(include=np.number)
    ).sum().sort_values(ascending=False)
)

Original thermal feature rows: 439251

Missing values:
observation_count      0
duration_minutes       0
mean_frp               0
peak_frp               0
total_frp              0
frp_range              0
frp_peak_ratio         0
frp_per_observation    0
mean_brightness        0
start_hour             0
day_of_week            0
month                  0
is_weekend             0
persistent             0
long_duration          0
is_day                 0
is_night               0
dtype: int64

Infinite values:
observation_count      0
duration_minutes       0
mean_frp               0
peak_frp               0
total_frp              0
frp_range              0
frp_peak_ratio         0
frp_per_observation    0
mean_brightness        0
start_hour             0
day_of_week            0
month                  0
is_weekend             0
persistent             0
long_duration          0
is_day                 0
is_night               0
dtype: int64


In [40]:
X_clean_check = X_check.replace([np.inf, -np.inf], np.nan).dropna()

print("Rows before cleaning:", len(X_check))
print("Rows after cleaning:", len(X_clean_check))
print("Rows removed:", len(X_check) - len(X_clean_check))

Rows before cleaning: 439251
Rows after cleaning: 439251
Rows removed: 0


In [41]:
removed_rows = X_check[
    X_check.replace([np.inf, -np.inf], np.nan)
    .isna()
    .any(axis=1)
]

print("Rows removed by dropna:", len(removed_rows))

Rows removed by dropna: 0


In [42]:
# Load the saved anomaly result
anomaly = pd.read_csv(
    PROJECT_ROOT / "data/results/thermal_anomaly_prototype.csv"
)

print("Anomaly file shape:", anomaly.shape)

print("\nColumns:")
print(anomaly.columns.tolist())

print("\nUnique event keys:")
print(anomaly[event_key].drop_duplicates().shape[0])

print("\nDuplicate event keys:")
print(anomaly.duplicated(event_key).sum())

Anomaly file shape: (439251, 25)

Columns:
['acq_date', 'start_time', 'end_time', 'latitude', 'longitude', 'observation_count', 'duration_minutes', 'mean_frp', 'peak_frp', 'total_frp', 'frp_range', 'frp_peak_ratio', 'frp_per_observation', 'mean_brightness', 'start_hour', 'day_of_week', 'month', 'is_weekend', 'persistent', 'long_duration', 'is_day', 'is_night', 'anomaly_score', 'anomaly_prediction', 'is_anomaly']

Unique event keys:
439251

Duplicate event keys:
0


In [43]:
# Compare event keys
event_keys = set(
    map(tuple, events[event_key].values)
)

anomaly_keys = set(
    map(tuple, anomaly[event_key].values)
)

missing_keys_set = event_keys - anomaly_keys
extra_keys_set = anomaly_keys - event_keys

print("Events:", len(event_keys))
print("Anomaly events:", len(anomaly_keys))
print("Missing from anomaly:", len(missing_keys_set))
print("Extra anomaly events:", len(extra_keys_set))

Events: 439251
Anomaly events: 439251
Missing from anomaly: 439251
Extra anomaly events: 439251


In [44]:
from pathlib import Path

model_path = PROJECT_ROOT / "model/isolation_forest_prototype.joblib"
scaler_path = PROJECT_ROOT / "model/scaler_prototype.joblib"

print("Model exists:", model_path.exists())
print("Scaler exists:", scaler_path.exists())

Model exists: True
Scaler exists: True


In [45]:
thermal_features.loc[
    thermal_features["frp_peak_ratio"].isna(),
    [
        "observation_count",
        "mean_frp",
        "peak_frp",
        "total_frp",
        "frp_range",
        "frp_peak_ratio",
        "frp_per_observation"
    ]
]

,observation_count,mean_frp,peak_frp,total_frp,frp_range,frp_peak_ratio,frp_per_observation


In [46]:
print(
    "Missing frp_peak_ratio:",
    thermal_features["frp_peak_ratio"].isna().sum()
)

print(
    "Infinite frp_peak_ratio:",
    np.isinf(thermal_features["frp_peak_ratio"]).sum()
)

Missing frp_peak_ratio: 0
Infinite frp_peak_ratio: 0


In [47]:
thermal_features.to_csv(
    PROJECT_ROOT / "data/features/thermal_features_prototype.csv",
    index=False
)

In [48]:
anomaly = pd.read_csv(
    PROJECT_ROOT / "data/results/thermal_anomaly_prototype.csv"
)

for df in [anomaly]:
    df["acq_date"] = pd.to_datetime(df["acq_date"], errors="coerce")

print("Anomaly shape:", anomaly.shape)

print("\nAnomaly columns:")
print(anomaly.columns.tolist())

print("\nDuplicate event keys:")
print(anomaly.duplicated(event_key).sum())

Anomaly shape: (439251, 25)

Anomaly columns:
['acq_date', 'start_time', 'end_time', 'latitude', 'longitude', 'observation_count', 'duration_minutes', 'mean_frp', 'peak_frp', 'total_frp', 'frp_range', 'frp_peak_ratio', 'frp_per_observation', 'mean_brightness', 'start_hour', 'day_of_week', 'month', 'is_weekend', 'persistent', 'long_duration', 'is_day', 'is_night', 'anomaly_score', 'anomaly_prediction', 'is_anomaly']

Duplicate event keys:
0


In [49]:
event_keys = set(
    map(tuple, events[event_key].values)
)

anomaly_keys = set(
    map(tuple, anomaly[event_key].values)
)

missing_keys_set = event_keys - anomaly_keys

print("Total events:", len(event_keys))
print("Anomaly events:", len(anomaly_keys))
print("Missing anomaly events:", len(missing_keys_set))

Total events: 439251
Anomaly events: 439251
Missing anomaly events: 439251


In [50]:
import joblib

model_path = PROJECT_ROOT / "model/isolation_forest_prototype.joblib"
scaler_path = PROJECT_ROOT / "model/scaler_prototype.joblib"

print("Model exists:", model_path.exists())
print("Scaler exists:", scaler_path.exists())

model = joblib.load(model_path)
scaler = joblib.load(scaler_path)

print("Model:", type(model))
print("Scaler:", type(scaler))

Model exists: True
Scaler exists: True
Model: <class 'sklearn.ensemble._iforest.IsolationForest'>
Scaler: <class 'sklearn.preprocessing._data.StandardScaler'>


In [51]:
print("Model feature count:", model.n_features_in_)
print("Scaler feature count:", scaler.n_features_in_)

Model feature count: 17
Scaler feature count: 17


In [52]:
# ============================================================
# RECREATE THE 382 MISSING EVENTS DIRECTLY FROM EVENTS
# ============================================================

# Make copies
events_check = events.copy()
anomaly_check = anomaly.copy()
features_check = thermal_features.copy()

# Standardize datetime columns
for df in [events_check, anomaly_check, features_check]:

    df["acq_date"] = pd.to_datetime(
        df["acq_date"], errors="coerce"
    )

    df["start_time"] = pd.to_datetime(
        df["start_time"], errors="coerce"
    )

    df["end_time"] = pd.to_datetime(
        df["end_time"], errors="coerce"
    )

# Standardize coordinates
for df in [events_check, anomaly_check, features_check]:

    df["latitude"] = pd.to_numeric(
        df["latitude"], errors="coerce"
    )

    df["longitude"] = pd.to_numeric(
        df["longitude"], errors="coerce"
    )


# ------------------------------------------------------------
# Find events that exist in events but NOT in anomaly
# ------------------------------------------------------------

events_with_flag = events_check.merge(
    anomaly_check[event_key].drop_duplicates(),
    on=event_key,
    how="left",
    indicator=True
)

missing_events = events_with_flag[
    events_with_flag["_merge"] == "left_only"
].drop(columns="_merge")

print("Total events:", len(events_check))
print("Anomaly events:", len(anomaly_check))
print("Missing events:", len(missing_events))

Total events: 439251
Anomaly events: 439251
Missing events: 382


In [53]:
missing_event_features = missing_events.merge(
    features_check,
    on=event_key,
    how="inner",
    suffixes=("", "_feature")
)

print(
    "Missing events matched with thermal features:",
    len(missing_event_features)
)

Missing events matched with thermal features: 0


In [54]:
ml_features = [
    "observation_count",
    "duration_minutes",
    "mean_frp",
    "peak_frp",
    "total_frp",
    "frp_range",
    "frp_peak_ratio",
    "frp_per_observation",
    "mean_brightness",
    "start_hour",
    "day_of_week",
    "month",
    "is_weekend",
    "persistent",
    "long_duration",
    "is_day",
    "is_night"
]

X_missing = missing_event_features[ml_features].copy()

print("Shape:", X_missing.shape)

print("\nMissing values:")
print(X_missing.isna().sum().sort_values(ascending=False))

print("\nInfinite values:")
print(
    np.isinf(
        X_missing.select_dtypes(include=np.number)
    ).sum().sum()
)

Shape: (0, 17)

Missing values:
observation_count      0
duration_minutes       0
mean_frp               0
peak_frp               0
total_frp              0
frp_range              0
frp_peak_ratio         0
frp_per_observation    0
mean_brightness        0
start_hour             0
day_of_week            0
month                  0
is_weekend             0
persistent             0
long_duration          0
is_day                 0
is_night               0
dtype: int64

Infinite values:
0


In [55]:
print("Anomaly rows:", len(anomaly_check))

print(
    "Unique anomaly events:",
    anomaly_check[event_key].drop_duplicates().shape[0]
)

print(
    "Duplicate anomaly event keys:",
    anomaly_check.duplicated(event_key).sum()
)

Anomaly rows: 439251
Unique anomaly events: 439251
Duplicate anomaly event keys: 0


In [56]:
duplicate_counts = (
    anomaly_check
    .groupby(event_key)
    .size()
    .reset_index(name="count")
)

print(
    duplicate_counts["count"]
    .value_counts()
    .sort_index()
)

count
1    439251
Name: count, dtype: int64


In [57]:
print(
    missing_events[
        event_key
    ].head(20).to_string(index=False)
)

  acq_date          start_time            end_time  latitude  longitude
2025-01-01 2025-01-01 07:16:00 2025-01-01 07:41:00 21.057533  81.165493
2025-01-01 2025-01-01 19:46:00 2025-01-01 20:55:00 21.078748  85.035886
2025-01-01 2025-01-01 07:16:00 2025-01-01 08:04:00 22.042246  83.734661
2025-01-01 2025-01-01 07:16:00 2025-01-01 08:04:00 22.122003  83.476057
2025-01-02 2025-01-02 19:25:00 2025-01-02 20:14:00 20.806923  85.256619
2025-01-02 2025-01-02 19:25:00 2025-01-02 19:51:00 22.369753  87.303030
2025-01-02 2025-01-02 19:25:00 2025-01-02 20:13:00 23.674463  86.086627
2025-01-02 2025-01-02 19:25:00 2025-01-02 20:13:00 23.780933  86.373380
2025-01-03 2025-01-03 19:08:00 2025-01-03 19:55:00 20.787266  85.264079
2025-01-03 2025-01-03 19:08:00 2025-01-03 19:55:00 21.078098  85.035876
2025-01-03 2025-01-03 19:32:00 2025-01-03 19:55:00 21.922773  83.350670
2025-01-04 2025-01-04 07:07:00 2025-01-04 07:07:00 21.101923  81.642307
2025-01-04 2025-01-04 06:22:00 2025-01-04 06:45:00 26.760103  95

In [58]:
# ============================================================
# DEFINITIVE DUPLICATE CHECK
# ============================================================

print("Anomaly rows:", len(anomaly_check))

unique_anomaly_keys = anomaly_check[event_key].drop_duplicates()

print("Unique anomaly event keys:", len(unique_anomaly_keys))

print(
    "Duplicate anomaly rows:",
    anomaly_check.duplicated(event_key).sum()
)

# How many times does each event key occur?
duplicate_counts = (
    anomaly_check
    .groupby(event_key)
    .size()
    .reset_index(name="count")
)

print("\nDistribution of event-key counts:")
print(
    duplicate_counts["count"]
    .value_counts()
    .sort_index()
)

print("\nEvent keys occurring more than once:")
print(
    duplicate_counts[
        duplicate_counts["count"] > 1
    ].head(20).to_string(index=False)
)

Anomaly rows: 439251
Unique anomaly event keys: 439251
Duplicate anomaly rows: 0

Distribution of event-key counts:
count
1    439251
Name: count, dtype: int64

Event keys occurring more than once:
Empty DataFrame
Columns: [acq_date, start_time, end_time, latitude, longitude, count]
Index: []


In [59]:
# Fix 0/0 cases
thermal_features["frp_peak_ratio"] = np.where(
    thermal_features["mean_frp"] > 0,
    thermal_features["peak_frp"] / thermal_features["mean_frp"],
    0.0
)

print("NaN:", thermal_features["frp_peak_ratio"].isna().sum())
print("Inf:", np.isinf(thermal_features["frp_peak_ratio"]).sum())

NaN: 0
Inf: 0


In [60]:
thermal_features.to_csv(
    PROJECT_ROOT / "data/features/thermal_features_prototype.csv",
    index=False
)

In [61]:
ml_features = [
    "observation_count",
    "duration_minutes",
    "mean_frp",
    "peak_frp",
    "total_frp",
    "frp_range",
    "frp_peak_ratio",
    "frp_per_observation",
    "mean_brightness",
    "start_hour",
    "day_of_week",
    "month",
    "is_weekend",
    "persistent",
    "long_duration",
    "is_day",
    "is_night"
]

X_all = thermal_features[ml_features].copy()

print("Feature matrix:", X_all.shape)
print("NaN:", X_all.isna().sum().sum())
print(
    "Inf:",
    np.isinf(X_all.select_dtypes(include=np.number)).sum().sum()
)

Feature matrix: (439251, 17)
NaN: 0
Inf: 0


In [62]:
X_all_scaled = scaler.transform(X_all)

pred_all = model.predict(X_all_scaled)
score_all = model.decision_function(X_all_scaled)

print("Predictions:", len(pred_all))
print("Scores:", len(score_all))

Predictions: 439251
Scores: 439251


In [63]:
# ============================================================
# REBUILD COMPLETE ANOMALY DATASET
# ============================================================

import numpy as np
import pandas as pd
import joblib

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

model_path = PROJECT_ROOT / "model/isolation_forest_prototype.joblib"
scaler_path = PROJECT_ROOT / "model/scaler_prototype.joblib"

# ------------------------------------------------------------
# 2. Load existing model + scaler
# ------------------------------------------------------------

model = joblib.load(model_path)
scaler = joblib.load(scaler_path)

print("Model and scaler loaded.")


# ------------------------------------------------------------
# 3. Reload corrected thermal features
# ------------------------------------------------------------

thermal_features = pd.read_csv(
    PROJECT_ROOT / "data/features/thermal_features_prototype.csv"
)

print("Thermal feature rows:", len(thermal_features))


# ------------------------------------------------------------
# 4. Fix FRP peak ratio
# ------------------------------------------------------------

thermal_features["frp_peak_ratio"] = np.where(
    thermal_features["mean_frp"] > 0,
    thermal_features["peak_frp"] /
    thermal_features["mean_frp"],
    0.0
)


# ------------------------------------------------------------
# 5. Define exact 17 model features
# ------------------------------------------------------------

ml_features = [
    "observation_count",
    "duration_minutes",
    "mean_frp",
    "peak_frp",
    "total_frp",
    "frp_range",
    "frp_peak_ratio",
    "frp_per_observation",
    "mean_brightness",
    "start_hour",
    "day_of_week",
    "month",
    "is_weekend",
    "persistent",
    "long_duration",
    "is_day",
    "is_night"
]


# ------------------------------------------------------------
# 6. Create feature matrix
# ------------------------------------------------------------

X_all = thermal_features[ml_features].copy()

print("Feature matrix:", X_all.shape)

print(
    "Missing values:",
    X_all.isna().sum().sum()
)

print(
    "Infinite values:",
    np.isinf(
        X_all.select_dtypes(include=np.number)
    ).sum().sum()
)


# ------------------------------------------------------------
# 7. Scale using EXISTING scaler
# ------------------------------------------------------------

X_all_scaled = scaler.transform(X_all)


# ------------------------------------------------------------
# 8. Predict using EXISTING Isolation Forest
# ------------------------------------------------------------

pred_all = model.predict(X_all_scaled)

score_all = model.decision_function(X_all_scaled)


# ------------------------------------------------------------
# 9. Create complete anomaly dataset
# ------------------------------------------------------------

anomaly_complete = thermal_features.copy()

anomaly_complete["anomaly_prediction"] = pred_all

anomaly_complete["anomaly_score"] = score_all

anomaly_complete["is_anomaly"] = (
    anomaly_complete["anomaly_prediction"] == -1
)


# ------------------------------------------------------------
# 10. Verify
# ------------------------------------------------------------

print("\n========== FINAL CHECK ==========")

print(
    "Total rows:",
    len(anomaly_complete)
)

print(
    "Unique event rows:",
    anomaly_complete[
        event_key
    ].drop_duplicates().shape[0]
)

print(
    "Duplicate event keys:",
    anomaly_complete.duplicated(
        event_key
    ).sum()
)

print("\nPrediction distribution:")

print(
    anomaly_complete[
        "anomaly_prediction"
    ].value_counts()
)

print("\nAnomaly count:")

print(
    anomaly_complete[
        "is_anomaly"
    ].value_counts()
)

Model and scaler loaded.
Thermal feature rows: 439251
Feature matrix: (439251, 17)
Missing values: 0
Infinite values: 0

========== FINAL CHECK ==========
Total rows: 439251
Unique event rows: 439251
Duplicate event keys: 0

Prediction distribution:
anomaly_prediction
 1    356761
-1     82490
Name: count, dtype: int64

Anomaly count:
is_anomaly
False    356761
True      82490
Name: count, dtype: int64


In [64]:
new_anomaly_path = (
    PROJECT_ROOT /
    "data/results/thermal_anomaly_prototype_rebuilt.csv"
)

anomaly_complete.to_csv(
    new_anomaly_path,
    index=False
)

print("Saved successfully:")
print(new_anomaly_path)

Saved successfully:
C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\data\results\thermal_anomaly_prototype_rebuilt.csv


In [65]:
events = pd.read_csv(
    PROJECT_ROOT / "data/events/thermal_events_prototype.csv"
)

anomaly = pd.read_csv(
    PROJECT_ROOT /
    "data/results/thermal_anomaly_prototype_rebuilt.csv"
)

context = pd.read_csv(
    PROJECT_ROOT /
    "data/features/firms_context_features_time_aware.csv"
)

events["acq_date"] = pd.to_datetime(
    events["acq_date"], errors="coerce"
)

anomaly["acq_date"] = pd.to_datetime(
    anomaly["acq_date"], errors="coerce"
)

context["timestamp"] = pd.to_datetime(
    context["timestamp"], errors="coerce"
)

print("Events:", events.shape)
print("Anomaly:", anomaly.shape)
print("Context:", context.shape)

Events: (439251, 14)
Anomaly: (439251, 25)
Context: (702968, 9)


In [66]:
event_key = [
    "acq_date",
    "start_time",
    "end_time",
    "latitude",
    "longitude"
]

for df in [events, anomaly]:

    df["start_time"] = pd.to_datetime(
        df["start_time"],
        errors="coerce"
    )

    df["end_time"] = pd.to_datetime(
        df["end_time"],
        errors="coerce"
    )

print(
    "Event unique keys:",
    events[event_key].drop_duplicates().shape[0]
)

print(
    "Anomaly unique keys:",
    anomaly[event_key].drop_duplicates().shape[0]
)

print(
    "Anomaly duplicate keys:",
    anomaly.duplicated(event_key).sum()
)

Event unique keys: 439251
Anomaly unique keys: 439251
Anomaly duplicate keys: 0


In [67]:
new_anomaly_path = (
    PROJECT_ROOT /
    "data/results/thermal_anomaly_prototype_rebuilt.csv"
)

anomaly_complete.to_csv(
    new_anomaly_path,
    index=False
)

print("Saved successfully:")
print(new_anomaly_path)

Saved successfully:
C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\data\results\thermal_anomaly_prototype_rebuilt.csv


In [68]:
events = pd.read_csv(
    PROJECT_ROOT / "data/events/thermal_events_prototype.csv"
)

anomaly = pd.read_csv(
    PROJECT_ROOT /
    "data/results/thermal_anomaly_prototype_rebuilt.csv"
)

context = pd.read_csv(
    PROJECT_ROOT /
    "data/features/firms_context_features_time_aware.csv"
)

print("Events:", events.shape)
print("Anomaly:", anomaly.shape)
print("Context:", context.shape)

Events: (439251, 14)
Anomaly: (439251, 25)
Context: (702968, 9)


In [69]:
for df in [events, anomaly]:

    df["acq_date"] = pd.to_datetime(
        df["acq_date"],
        errors="coerce"
    )

    df["start_time"] = pd.to_datetime(
        df["start_time"],
        errors="coerce"
    )

    df["end_time"] = pd.to_datetime(
        df["end_time"],
        errors="coerce"
    )

context["timestamp"] = pd.to_datetime(
    context["timestamp"],
    errors="coerce"
)

context["acq_date"] = context["timestamp"].dt.normalize()

In [70]:
event_key = [
    "acq_date",
    "start_time",
    "end_time",
    "latitude",
    "longitude"
]

print(
    "Events:",
    events[event_key].drop_duplicates().shape[0]
)

print(
    "Anomaly:",
    anomaly[event_key].drop_duplicates().shape[0]
)

print(
    "Anomaly duplicate keys:",
    anomaly.duplicated(event_key).sum()
)

Events: 439251
Anomaly: 439251
Anomaly duplicate keys: 0


In [71]:
# ============================================================
# PREPARE CONTEXT FEATURES
# ============================================================

SPATIAL_CELL_KM = 1.0

LAT_CELL = SPATIAL_CELL_KM / 111.0
LON_CELL = SPATIAL_CELL_KM / 100.0

context["lat_cell"] = np.floor(
    context["latitude"] / LAT_CELL
).astype("int32")

context["lon_cell"] = np.floor(
    context["longitude"] / LON_CELL
).astype("int32")

print("Context rows:", len(context))
print("Unique context observations:", context["timestamp"].nunique())

Context rows: 702968
Unique context observations: 5902


In [72]:
context_event = (
    context
    .groupby(
        ["acq_date", "lat_cell", "lon_cell"],
        as_index=False
    )
    .agg(
        historical_detection_count=(
            "historical_detection_count",
            "max"
        ),
        historical_mean_frp=(
            "historical_mean_frp",
            "max"
        ),
        local_frp_deviation=(
            "local_frp_deviation",
            "max"
        ),
        historical_daily_activity=(
            "historical_daily_activity",
            "max"
        ),
        previously_detected=(
            "previously_detected",
            "max"
        )
    )
)

print("Aggregated context:", context_event.shape)

print(
    context_event[
        [
            "acq_date",
            "lat_cell",
            "lon_cell"
        ]
    ].duplicated().sum()
)

Aggregated context: (382881, 8)
0


In [73]:
events["lat_cell"] = np.floor(
    events["latitude"] / LAT_CELL
).astype("int32")

events["lon_cell"] = np.floor(
    events["longitude"] / LON_CELL
).astype("int32")

In [74]:
event_context = events.merge(
    context_event,
    on=["acq_date", "lat_cell", "lon_cell"],
    how="left",
    validate="many_to_one"
)

print("Event-context shape:", event_context.shape)

print(
    "Rows changed:",
    len(event_context) - len(events)
)

Event-context shape: (439251, 19)
Rows changed: 0


In [75]:
event_context_anomaly = event_context.merge(
    anomaly[
        event_key +
        [
            "anomaly_prediction",
            "anomaly_score",
            "is_anomaly"
        ]
    ],
    on=event_key,
    how="left",
    validate="one_to_one"
)

print(
    "Final event-level shape:",
    event_context_anomaly.shape
)


Final event-level shape: (439251, 22)


In [76]:
print(
    "Rows:",
    len(event_context_anomaly)
)

print(
    "Unique events:",
    event_context_anomaly[event_key]
    .drop_duplicates()
    .shape[0]
)

print(
    "Duplicate events:",
    event_context_anomaly
    .duplicated(event_key)
    .sum()
)

print(
    "\nMissing anomaly predictions:",
    event_context_anomaly["anomaly_prediction"]
    .isna()
    .sum()
)

Rows: 439251
Unique events: 439251
Duplicate events: 0

Missing anomaly predictions: 382


In [78]:
# ============================================================
# TEMPORAL CONTEXT AUDIT — ONE COMPLETE CELL
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Load the original time-aware context
# ------------------------------------------------------------

context_audit = pd.read_csv(
    PROJECT_ROOT /
    "data/features/firms_context_features_time_aware.csv"
)

# ------------------------------------------------------------
# 2. Convert datetime
# ------------------------------------------------------------

context_audit["timestamp"] = pd.to_datetime(
    context_audit["timestamp"],
    errors="coerce"
)

context_audit["acq_date"] = (
    context_audit["timestamp"].dt.normalize()
)

# ------------------------------------------------------------
# 3. Recreate spatial grid
# ------------------------------------------------------------

SPATIAL_CELL_KM = 1.0

LAT_CELL = SPATIAL_CELL_KM / 111.0
LON_CELL = SPATIAL_CELL_KM / 100.0

context_audit["lat_cell"] = np.floor(
    context_audit["latitude"] / LAT_CELL
).astype("int32")

context_audit["lon_cell"] = np.floor(
    context_audit["longitude"] / LON_CELL
).astype("int32")

# ------------------------------------------------------------
# 4. Sort chronologically
# ------------------------------------------------------------

context_audit = context_audit.sort_values(
    ["lat_cell", "lon_cell", "timestamp"]
).reset_index(drop=True)

print("Context rows:", len(context_audit))

# ------------------------------------------------------------
# 5. Inspect one spatial cell
# ------------------------------------------------------------

sample_cell = (
    context_audit[
        ["lat_cell", "lon_cell"]
    ]
    .drop_duplicates()
    .iloc[0]
)

sample = context_audit[
    (context_audit["lat_cell"] == sample_cell["lat_cell"]) &
    (context_audit["lon_cell"] == sample_cell["lon_cell"])
].copy()

print("\nSample spatial cell:")
print(
    "lat_cell:",
    sample_cell["lat_cell"]
)

print(
    "lon_cell:",
    sample_cell["lon_cell"]
)

print("\nFirst observations:")

print(
    sample[
        [
            "timestamp",
            "frp",
            "historical_detection_count",
            "historical_mean_frp",
            "local_frp_deviation",
            "historical_daily_activity",
            "previously_detected"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 6. Same-day aggregation used in Notebook 08
# ------------------------------------------------------------

daily_context = (
    context_audit
    .groupby(
        ["acq_date", "lat_cell", "lon_cell"],
        as_index=False
    )
    .agg(
        max_historical_detection=(
            "historical_detection_count",
            "max"
        ),
        max_historical_daily_activity=(
            "historical_daily_activity",
            "max"
        ),
        max_historical_mean_frp=(
            "historical_mean_frp",
            "max"
        ),
        max_local_frp_deviation=(
            "local_frp_deviation",
            "max"
        )
    )
)

print("\nSame-day/grid aggregated context:")
print(daily_context.shape)

print(
    daily_context.head(20).to_string(index=False)
)

Context rows: 702968

Sample spatial cell:
lat_cell: 896
lon_cell: 7753

First observations:
          timestamp  frp  historical_detection_count  historical_mean_frp  local_frp_deviation  historical_daily_activity  previously_detected
2025-01-11 20:26:00 2.60                           0                0.000                2.600                          0                    0
2025-01-12 20:30:00 1.34                           1                2.600               -1.260                          1                    1
2025-01-17 20:37:00 0.98                           2                1.970               -0.990                          2                    1
2025-03-01 19:42:00 0.86                           3                1.640               -0.780                          3                    1
2025-03-01 20:08:00 1.18                           4                1.445               -0.265                          3                    1

Same-day/grid aggregated context:
(382881, 7)
  

In [ ]:
print(
    "Rows:",
    len(event_context_anomaly)
)

print(
    "Unique events:",
    event_context_anomaly[event_key]
    .drop_duplicates()
    .shape[0]
)

print(
    "Duplicate events:",
    event_context_anomaly
    .duplicated(event_key)
    .sum()
)

print(
    "\nMissing anomaly predictions:",
    event_context_anomaly["anomaly_prediction"]
    .isna()
    .sum()
)

Rows: 439251
Unique events: 439251
Duplicate events: 0

Missing anomaly predictions: 382


In [82]:
# ============================================================
# FAST LEAKAGE-SAFE EVENT CONTEXT
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------

events_final = pd.read_csv(
    PROJECT_ROOT / "data/events/thermal_events_prototype.csv"
)

anomaly_final = pd.read_csv(
    PROJECT_ROOT / "data/results/thermal_anomaly_prototype_rebuilt.csv"
)

context_final = pd.read_csv(
    PROJECT_ROOT / "data/features/firms_context_features_time_aware.csv"
)

# ------------------------------------------------------------
# 2. Datetime conversion
# ------------------------------------------------------------

for df in [events_final, anomaly_final]:
    df["acq_date"] = pd.to_datetime(df["acq_date"], errors="coerce")
    df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")
    df["end_time"] = pd.to_datetime(df["end_time"], errors="coerce")

context_final["timestamp"] = pd.to_datetime(
    context_final["timestamp"],
    errors="coerce"
)

# ------------------------------------------------------------
# 3. Spatial grid
# ------------------------------------------------------------

SPATIAL_CELL_KM = 1.0

LAT_CELL = SPATIAL_CELL_KM / 111.0
LON_CELL = SPATIAL_CELL_KM / 100.0

events_final["lat_cell"] = np.floor(
    events_final["latitude"] / LAT_CELL
).astype("int32")

events_final["lon_cell"] = np.floor(
    events_final["longitude"] / LON_CELL
).astype("int32")

context_final["lat_cell"] = np.floor(
    context_final["latitude"] / LAT_CELL
).astype("int32")

context_final["lon_cell"] = np.floor(
    context_final["longitude"] / LON_CELL
).astype("int32")

# ------------------------------------------------------------
# 4. Keep only required context columns
# ------------------------------------------------------------

context_lookup = context_final[
    [
        "lat_cell",
        "lon_cell",
        "timestamp",
        "historical_detection_count",
        "historical_mean_frp",
        "local_frp_deviation",
        "historical_daily_activity",
        "previously_detected"
    ]
].copy()

# ------------------------------------------------------------
# 5. IMPORTANT:
# merge_asof requires the TIME column to be sorted globally
# ------------------------------------------------------------

events_final = events_final.sort_values(
    ["start_time", "lat_cell", "lon_cell"]
).reset_index(drop=True)

context_lookup = context_lookup.sort_values(
    ["timestamp", "lat_cell", "lon_cell"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 6. Fast leakage-safe merge
# ------------------------------------------------------------

event_context_final = pd.merge_asof(
    events_final,
    context_lookup,
    left_on="start_time",
    right_on="timestamp",
    left_by=["lat_cell", "lon_cell"],
    right_by=["lat_cell", "lon_cell"],
    direction="backward",
    allow_exact_matches=False
)

# ------------------------------------------------------------
# 7. Check context
# ------------------------------------------------------------

context_cols = [
    "historical_detection_count",
    "historical_mean_frp",
    "local_frp_deviation",
    "historical_daily_activity",
    "previously_detected"
]

print("========== CONTEXT CHECK ==========")

print("Rows:", len(event_context_final))

print(
    "Missing historical context:",
    event_context_final[
        "historical_detection_count"
    ].isna().sum()
)

print("\nContext preview:")

print(
    event_context_final[
        [
            "start_time",
            "latitude",
            "longitude"
        ] + context_cols
    ].head(10)
)

========== CONTEXT CHECK ==========
Rows: 439251
Missing historical context: 257899

Context preview:
           start_time  latitude  longitude  historical_detection_count  \
0 2025-01-01 03:50:00   26.9361    94.0335                         NaN   
1 2025-01-01 03:51:00   25.3231    91.6008                         NaN   
2 2025-01-01 03:51:00   25.3249    91.5900                         NaN   
3 2025-01-01 03:51:00   26.2760    92.8154                         NaN   
4 2025-01-01 03:51:00   26.7587    92.7613                         NaN   
5 2025-01-01 03:52:00   20.9661    85.1728                         NaN   
6 2025-01-01 03:52:00   21.0756    85.0339                         NaN   
7 2025-01-01 03:52:00   21.0753    85.0405                         NaN   
8 2025-01-01 03:52:00   22.0373    83.7321                         NaN   
9 2025-01-01 03:52:00   22.3815    87.2787                         NaN   

   historical_mean_frp  local_frp_deviation  historical_daily_activity  \
0        

In [83]:
# ============================================================
# DIAGNOSE MISSING HISTORICAL CONTEXT
# ============================================================

context_cols = [
    "historical_detection_count",
    "historical_mean_frp",
    "local_frp_deviation",
    "historical_daily_activity",
    "previously_detected"
]

# ------------------------------------------------------------
# 1. Events with / without historical context
# ------------------------------------------------------------

has_context = event_context_final[
    "historical_detection_count"
].notna()

print("Total events:", len(event_context_final))
print("With historical context:", has_context.sum())
print("Without historical context:", (~has_context).sum())

print(
    "Context coverage:",
    round(has_context.mean() * 100, 2),
    "%"
)

# ------------------------------------------------------------
# 2. Check whether missing context is mainly early data
# ------------------------------------------------------------

event_context_final["event_date"] = (
    event_context_final["start_time"].dt.date
)

daily_missing = (
    event_context_final
    .groupby("event_date")
    .agg(
        total_events=("start_time", "size"),
        missing_context=(
            "historical_detection_count",
            lambda x: x.isna().sum()
        )
    )
    .reset_index()
)

daily_missing["missing_percentage"] = (
    daily_missing["missing_context"]
    / daily_missing["total_events"]
    * 100
)

print("\nFirst 15 dates:")

print(
    daily_missing.head(15)
)

# ------------------------------------------------------------
# 3. Check last 15 dates
# ------------------------------------------------------------

print("\nLast 15 dates:")

print(
    daily_missing.tail(15)
)

# ------------------------------------------------------------
# 4. Check context availability by event
# ------------------------------------------------------------

print("\nHistorical context distribution:")

print(
    event_context_final[
        "historical_detection_count"
    ].describe()
)

# ------------------------------------------------------------
# 5. Sample events without context
# ------------------------------------------------------------

print("\nSample events WITHOUT historical context:")

print(
    event_context_final.loc[
        ~has_context,
        [
            "start_time",
            "latitude",
            "longitude",
            "lat_cell",
            "lon_cell",
            "mean_frp",
            "peak_frp"
        ]
    ].head(20)
)

Total events: 439251
With historical context: 181352
Without historical context: 257899
Context coverage: 41.29 %

First 15 dates:
    event_date  total_events  missing_context  missing_percentage
0   2025-01-01          2548             2309           90.620094
1   2025-01-02          2970             2194           73.872054
2   2025-01-03          3629             2398           66.078810
3   2025-01-04          3380             2293           67.840237
4   2025-01-05          2752             1812           65.843023
5   2025-01-06          2501             1621           64.814074
6   2025-01-07          2346             1526           65.046888
7   2025-01-08          2954             1796           60.798917
8   2025-01-09          2815             1636           58.117229
9   2025-01-10          2543             1518           59.693276
10  2025-01-11          2393             1373           57.375679
11  2025-01-12          1747             1010           57.813394
12  2025-01

In [84]:
# ============================================================
# CHECK WHETHER MISSING 1-KM HISTORY IS LEGITIMATE
# ============================================================

# Use the original time-aware context
ctx = context_final[
    [
        "timestamp",
        "latitude",
        "longitude",
        "lat_cell",
        "lon_cell",
        "historical_detection_count"
    ]
].copy()

ctx = ctx.sort_values("timestamp").reset_index(drop=True)

missing_events = event_context_final[
    event_context_final["historical_detection_count"].isna()
].copy()

print("Missing-context events:", len(missing_events))

# ------------------------------------------------------------
# Check same spatial cell, regardless of exact time
# ------------------------------------------------------------

cell_history = (
    ctx.groupby(
        ["lat_cell", "lon_cell"]
    )
    .agg(
        first_observation=("timestamp", "min"),
        total_observations=("timestamp", "size")
    )
    .reset_index()
)

missing_check = missing_events.merge(
    cell_history,
    on=["lat_cell", "lon_cell"],
    how="left"
)

# A cell has previous data if its first observation
# happened before the current event.
missing_check["has_previous_cell_history"] = (
    missing_check["first_observation"]
    < missing_check["start_time"]
)

print("\n========== RESULT ==========")

print(
    "Missing-context events:",
    len(missing_check)
)

print(
    "Missing events where cell had earlier observations:",
    missing_check[
        "has_previous_cell_history"
    ].sum()
)

print(
    "Missing events where cell had NO earlier observations:",
    (
        ~missing_check[
            "has_previous_cell_history"
        ].fillna(False)
    ).sum()
)

print("\nPercentage breakdown:")

print(
    missing_check[
        "has_previous_cell_history"
    ]
    .value_counts(dropna=False, normalize=True)
    .mul(100)
    .round(2)
)

# ------------------------------------------------------------
# Show examples of suspicious cases
# ------------------------------------------------------------

suspicious = missing_check[
    missing_check["has_previous_cell_history"] == True
]

print(
    "\nSuspicious missing-context events:",
    len(suspicious)
)

print(
    suspicious[
        [
            "start_time",
            "latitude",
            "longitude",
            "lat_cell",
            "lon_cell",
            "first_observation",
            "total_observations"
        ]
    ].head(20)
)

Missing-context events: 257899

========== RESULT ==========
Missing-context events: 257899
Missing events where cell had earlier observations: 0
Missing events where cell had NO earlier observations: 257899

Percentage breakdown:
has_previous_cell_history
False    100.0
Name: proportion, dtype: float64

Suspicious missing-context events: 0
Empty DataFrame
Columns: [start_time, latitude, longitude, lat_cell, lon_cell, first_observation, total_observations]
Index: []


In [85]:
# ============================================================
# HANDLE LEGITIMATE FIRST-EVER EVENTS
# ============================================================

context_cols = [
    "historical_detection_count",
    "historical_mean_frp",
    "local_frp_deviation",
    "historical_daily_activity",
    "previously_detected"
]

# These NaNs mean:
# "There was no previous observation in this spatial cell."
# Therefore historical activity = 0.

for col in context_cols:
    event_context_final[col] = (
        event_context_final[col].fillna(0)
    )

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("========== CONTEXT FINAL CHECK ==========")

print(
    "Rows:",
    len(event_context_final)
)

print(
    "Missing context values:"
)

print(
    event_context_final[
        context_cols
    ].isna().sum()
)

print("\nContext statistics:")

print(
    event_context_final[
        context_cols
    ].describe()
)

========== CONTEXT FINAL CHECK ==========
Rows: 439251
Missing context values:
historical_detection_count    0
historical_mean_frp           0
local_frp_deviation           0
historical_daily_activity     0
previously_detected           0
dtype: int64

Context statistics:
       historical_detection_count  historical_mean_frp  local_frp_deviation  \
count               439251.000000        439251.000000        439251.000000   
mean                    13.006359             1.836145             0.828698   
std                     62.331592             9.513431            11.681289   
min                      0.000000             0.000000          -852.680000   
25%                      0.000000             0.000000             0.000000   
50%                      0.000000             0.000000             0.000000   
75%                      1.000000             1.062918             0.129264   
max                   1568.000000           855.260000          1210.350000   

       historic

In [86]:
# ============================================================
# FINAL ANOMALY MERGE
# ============================================================

event_key = [
    "acq_date",
    "start_time",
    "end_time",
    "latitude",
    "longitude"
]

# ------------------------------------------------------------
# Load rebuilt anomaly data
# ------------------------------------------------------------

anomaly_final = pd.read_csv(
    PROJECT_ROOT /
    "data/results/thermal_anomaly_prototype_rebuilt.csv"
)

for col in ["acq_date", "start_time", "end_time"]:
    anomaly_final[col] = pd.to_datetime(
        anomaly_final[col],
        errors="coerce"
    )

# ------------------------------------------------------------
# Verify anomaly dataset
# ------------------------------------------------------------

print("========== ANOMALY DATASET ==========")

print("Rows:", len(anomaly_final))

print(
    "Unique event keys:",
    anomaly_final[event_key]
    .drop_duplicates()
    .shape[0]
)

print(
    "Duplicate event keys:",
    anomaly_final.duplicated(event_key).sum()
)

# ------------------------------------------------------------
# Select only anomaly columns
# ------------------------------------------------------------

anomaly_subset = anomaly_final[
    event_key + [
        "anomaly_prediction",
        "anomaly_score",
        "is_anomaly"
    ]
].copy()

# ------------------------------------------------------------
# Merge
# ------------------------------------------------------------

final_ml = event_context_final.merge(
    anomaly_subset,
    on=event_key,
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

print("\n========== FINAL ML DATASET ==========")

print("Rows:", len(final_ml))

print(
    "Unique events:",
    final_ml[event_key]
    .drop_duplicates()
    .shape[0]
)

print(
    "Duplicate events:",
    final_ml.duplicated(event_key).sum()
)

print(
    "Missing anomaly predictions:",
    final_ml[
        "anomaly_prediction"
    ].isna().sum()
)

print(
    "Missing anomaly scores:",
    final_ml[
        "anomaly_score"
    ].isna().sum()
)

print(
    "Missing anomaly flags:",
    final_ml[
        "is_anomaly"
    ].isna().sum()
)

print("\nAnomaly distribution:")

print(
    final_ml["is_anomaly"].value_counts()
)

========== ANOMALY DATASET ==========
Rows: 439251
Unique event keys: 439251
Duplicate event keys: 0

========== FINAL ML DATASET ==========
Rows: 439251
Unique events: 439251
Duplicate events: 0
Missing anomaly predictions: 382
Missing anomaly scores: 382
Missing anomaly flags: 382

Anomaly distribution:
is_anomaly
False    356687
True      82182
Name: count, dtype: int64


In [88]:
print("FINAL ML COLUMNS:")
for i, col in enumerate(final_ml.columns):
    print(i, col)

FINAL ML COLUMNS:
0 acq_date
1 lat_cell
2 lon_cell
3 event_number
4 start_time
5 end_time
6 observation_count
7 latitude
8 longitude
9 mean_frp
10 peak_frp
11 total_frp
12 mean_brightness
13 persistent
14 timestamp
15 historical_detection_count
16 historical_mean_frp
17 local_frp_deviation
18 historical_daily_activity
19 previously_detected
20 event_date
21 anomaly_prediction
22 anomaly_score
23 is_anomaly


In [89]:
# ============================================================
# SAFE FINAL ML DATASET AUDIT
# ============================================================

import numpy as np
import pandas as pd

print("========== BASIC CHECK ==========")

print("Rows:", len(final_ml))
print("Columns:", len(final_ml.columns))

print(
    "Unique events:",
    final_ml[event_key].drop_duplicates().shape[0]
)

print(
    "Duplicate events:",
    final_ml.duplicated(event_key).sum()
)

# ------------------------------------------------------------
# Missing values
# ------------------------------------------------------------

print("\n========== MISSING VALUES ==========")

missing_report = final_ml.isna().sum()
missing_report = missing_report[missing_report > 0]

if len(missing_report) == 0:
    print("No missing values.")
else:
    print(missing_report)

# ------------------------------------------------------------
# Infinite values
# ------------------------------------------------------------

print("\n========== INFINITE VALUES ==========")

numeric_cols = final_ml.select_dtypes(
    include=np.number
).columns

infinite_report = np.isinf(
    final_ml[numeric_cols]
).sum()

infinite_report = infinite_report[
    infinite_report > 0
]

if len(infinite_report) == 0:
    print("No infinite values.")
else:
    print(infinite_report)

# ------------------------------------------------------------
# Time check
# ------------------------------------------------------------

print("\n========== TIME CHECK ==========")

bad_time = (
    final_ml["start_time"] >
    final_ml["end_time"]
).sum()

print("Start > End:", bad_time)

print(
    "Date range:",
    final_ml["start_time"].min(),
    "→",
    final_ml["start_time"].max()
)

# ------------------------------------------------------------
# Anomaly check
# ------------------------------------------------------------

print("\n========== ANOMALY CHECK ==========")

print(
    final_ml["is_anomaly"].value_counts(
        dropna=False
    )
)

print(
    "Anomaly percentage:",
    round(
        final_ml["is_anomaly"].mean() * 100,
        2
    ),
    "%"
)

print(
    "Anomaly prediction values:",
    sorted(
        final_ml["anomaly_prediction"]
        .dropna()
        .unique()
        .tolist()
    )
)

print(
    "Anomaly score range:",
    final_ml["anomaly_score"].min(),
    "→",
    final_ml["anomaly_score"].max()
)

# ------------------------------------------------------------
# Safe value checks
# ------------------------------------------------------------

print("\n========== VALUE CHECK ==========")

checks = {
    "mean_frp": ("mean_frp", lambda x: x < 0),
    "peak_frp": ("peak_frp", lambda x: x < 0),
    "observation_count": ("observation_count", lambda x: x < 0),
    "duration_minutes": ("duration_minutes", lambda x: x < 0),
    "historical_detection_count": (
        "historical_detection_count",
        lambda x: x < 0
    ),
    "historical_daily_activity": (
        "historical_daily_activity",
        lambda x: x < 0
    )
}

for name, (column, condition) in checks.items():

    if column in final_ml.columns:
        count = condition(
            final_ml[column]
        ).sum()

        print(f"{name}: {count}")

    else:
        print(
            f"{name}: COLUMN NOT PRESENT"
        )

# ------------------------------------------------------------
# Available ML features
# ------------------------------------------------------------

print("\n========== AVAILABLE ML FEATURES ==========")

feature_candidates = [
    "observation_count",
    "duration_minutes",
    "mean_frp",
    "peak_frp",
    "total_frp",
    "frp_range",
    "frp_peak_ratio",
    "frp_per_observation",
    "mean_brightness",
    "start_hour",
    "day_of_week",
    "month",
    "is_weekend",
    "persistent",
    "long_duration",
    "is_day",
    "is_night",
    "historical_detection_count",
    "historical_mean_frp",
    "local_frp_deviation",
    "historical_daily_activity",
    "previously_detected",
    "has_historical_context"
]

available_features = [
    col
    for col in feature_candidates
    if col in final_ml.columns
]

missing_features = [
    col
    for col in feature_candidates
    if col not in final_ml.columns
]

print("\nAvailable:")
print(available_features)

print("\nMissing:")
print(missing_features)

# ------------------------------------------------------------
# Correlation — only available features
# ------------------------------------------------------------

print("\n========== CORRELATION ==========")

corr_features = [
    col
    for col in available_features
    if pd.api.types.is_numeric_dtype(
        final_ml[col]
    )
]

if len(corr_features) > 1:

    corr = (
        final_ml[corr_features]
        .corr()
        .round(3)
    )

    if "peak_frp" in corr.columns:
        print(
            corr["peak_frp"]
            .sort_values(ascending=False)
        )
    else:
        print(
            "peak_frp not available."
        )

# ------------------------------------------------------------
# Duplicate feature rows
# ------------------------------------------------------------

if len(corr_features) > 0:

    duplicate_feature_rows = (
        final_ml[corr_features]
        .duplicated()
        .sum()
    )

    print(
        "\nDuplicate feature rows:",
        duplicate_feature_rows
    )

print("\n========== AUDIT COMPLETE ==========")

========== BASIC CHECK ==========
Rows: 439251
Columns: 24
Unique events: 439251
Duplicate events: 0

========== MISSING VALUES ==========
timestamp             257899
anomaly_prediction       382
anomaly_score            382
is_anomaly               382
dtype: int64

========== INFINITE VALUES ==========
No infinite values.

========== TIME CHECK ==========
Start > End: 0
Date range: 2025-01-01 03:50:00 → 2025-03-19 21:30:00

========== ANOMALY CHECK ==========
is_anomaly
False    356687
True      82182
NaN         382
Name: count, dtype: int64
Anomaly percentage: 18.73 %
Anomaly prediction values: [-1.0, 1.0]
Anomaly score range: -0.3124409870597798 → 0.1366346446047436

========== VALUE CHECK ==========
mean_frp: 0
peak_frp: 0
observation_count: 0
duration_minutes: COLUMN NOT PRESENT
historical_detection_count: 0
historical_daily_activity: 0

========== AVAILABLE ML FEATURES ==========

Available:
['observation_count', 'mean_frp', 'peak_frp', 'total_frp', 'mean_brightness', 'persist

In [91]:
# ============================================================
# DIAGNOSE ANOMALY MERGE MISMATCH
# ============================================================

print("========== ANOMALY MERGE DIAGNOSTIC ==========")

# Keys from final event/context dataset
left_keys = event_context_final[event_key].copy()

# Keys from rebuilt anomaly dataset
right_keys = anomaly_final[event_key].copy()

# ------------------------------------------------------------
# Standardize datetime columns
# ------------------------------------------------------------

for col in ["acq_date", "start_time", "end_time"]:
    left_keys[col] = pd.to_datetime(
        left_keys[col],
        errors="coerce"
    )

    right_keys[col] = pd.to_datetime(
        right_keys[col],
        errors="coerce"
    )

# ------------------------------------------------------------
# Standardize coordinates
# ------------------------------------------------------------

for col in ["latitude", "longitude"]:
    left_keys[col] = pd.to_numeric(
        left_keys[col],
        errors="coerce"
    )

    right_keys[col] = pd.to_numeric(
        right_keys[col],
        errors="coerce"
    )

# ------------------------------------------------------------
# Find exact key matches
# ------------------------------------------------------------

matched = left_keys.merge(
    right_keys.drop_duplicates(),
    on=event_key,
    how="left",
    indicator=True
)

print("Total events:", len(matched))

print(
    "\nExact matches:",
    (matched["_merge"] == "both").sum()
)

print(
    "Unmatched:",
    (matched["_merge"] == "left_only").sum()
)

# ------------------------------------------------------------
# Show unmatched events
# ------------------------------------------------------------

unmatched = matched[
    matched["_merge"] == "left_only"
].drop(columns="_merge")

print("\nFirst unmatched events:")

display(
    unmatched.head(20)
)

========== ANOMALY MERGE DIAGNOSTIC ==========
Total events: 439251

Exact matches: 438869
Unmatched: 382

First unmatched events:


,acq_date,start_time,end_time,latitude,longitude
100,2025-01-01,2025-01-01 07:16:00,2025-01-01 07:41:00,21.057533,81.165493
390,2025-01-01,2025-01-01 07:16:00,2025-01-01 08:04:00,22.042246,83.734661
410,2025-01-01,2025-01-01 07:16:00,2025-01-01 08:04:00,22.122003,83.476057
2028,2025-01-01,2025-01-01 19:46:00,2025-01-01 20:55:00,21.078748,85.035886
4854,2025-01-02,2025-01-02 19:25:00,2025-01-02 20:14:00,20.806923,85.256619
4948,2025-01-02,2025-01-02 19:25:00,2025-01-02 19:51:00,22.369753,87.303030
4977,2025-01-02,2025-01-02 19:25:00,2025-01-02 20:13:00,23.674463,86.086627
5018,2025-01-02,2025-01-02 19:25:00,2025-01-02 20:13:00,23.780933,86.373380
8260,2025-01-03,2025-01-03 19:08:00,2025-01-03 19:55:00,20.787266,85.264079
8278,2025-01-03,2025-01-03 19:08:00,2025-01-03 19:55:00,21.078098,85.035876


In [92]:
# ============================================================
# FIND WHICH KEY DIFFERS
# ============================================================

print("========== KEY DIFFERENCE CHECK ==========")

# Take the first few unmatched rows
sample = unmatched.head(20)

for _, row in sample.iterrows():

    print("\n--------------------------------")

    print(
        "DATE:",
        row["acq_date"]
    )

    print(
        "START:",
        row["start_time"]
    )

    print(
        "END:",
        row["end_time"]
    )

    print(
        "LAT:",
        row["latitude"]
    )

    print(
        "LON:",
        row["longitude"]
    )

    # Find candidates with same date
    candidates = right_keys[
        right_keys["acq_date"] ==
        row["acq_date"]
    ]

    print(
        "Same-date anomaly rows:",
        len(candidates)
    )

    # Find same coordinates
    coord_candidates = candidates[
        np.isclose(
            candidates["latitude"],
            row["latitude"],
            atol=1e-8
        )
        &
        np.isclose(
            candidates["longitude"],
            row["longitude"],
            atol=1e-8
        )
    ]

    print(
        "Same-coordinate rows:",
        len(coord_candidates)
    )

    if len(coord_candidates) > 0:
        display(
            coord_candidates[
                [
                    "acq_date",
                    "start_time",
                    "end_time",
                    "latitude",
                    "longitude"
                ]
            ].head(5)
        )

========== KEY DIFFERENCE CHECK ==========

--------------------------------
DATE: 2025-01-01 00:00:00
START: 2025-01-01 07:16:00
END: 2025-01-01 07:41:00
LAT: 21.057533333333332
LON: 81.16549333333333
Same-date anomaly rows: 2548
Same-coordinate rows: 1


,acq_date,start_time,end_time,latitude,longitude
871,2025-01-01,2025-01-01 07:16:00,2025-01-01 07:41:00,21.057533,81.165493



--------------------------------
DATE: 2025-01-01 00:00:00
START: 2025-01-01 07:16:00
END: 2025-01-01 08:04:00
LAT: 22.042245714285716
LON: 83.73466142857144
Same-date anomaly rows: 2548
Same-coordinate rows: 1


,acq_date,start_time,end_time,latitude,longitude
1777,2025-01-01,2025-01-01 07:16:00,2025-01-01 08:04:00,22.042246,83.734661



--------------------------------
DATE: 2025-01-01 00:00:00
START: 2025-01-01 07:16:00
END: 2025-01-01 08:04:00
LAT: 22.122003333333332
LON: 83.47605666666668
Same-date anomaly rows: 2548
Same-coordinate rows: 1


...

,acq_date,start_time,end_time,latitude,longitude
17282,2025-01-06,2025-01-06 19:51:00,2025-01-06 20:56:00,23.680758,86.395213



--------------------------------
DATE: 2025-01-06 00:00:00
START: 2025-01-06 19:51:00
END: 2025-01-06 20:38:00
LAT: 23.687178333333332
LON: 86.39248833333333
Same-date anomaly rows: 2501
Same-coordinate rows: 1


,acq_date,start_time,end_time,latitude,longitude
17290,2025-01-06,2025-01-06 19:51:00,2025-01-06 20:38:00,23.687178,86.392488



--------------------------------
DATE: 2025-01-06 00:00:00
START: 2025-01-06 19:53:00
END: 2025-01-06 20:56:00
LAT: 20.795215714285717
LON: 85.25423142857143
Same-date anomaly rows: 2501
Same-coordinate rows: 1


,acq_date,start_time,end_time,latitude,longitude
16163,2025-01-06,2025-01-06 19:53:00,2025-01-06 20:56:00,20.795216,85.254231


In [94]:
# ============================================================
# PHASE 1 — FINAL SAVE
# ============================================================

import numpy as np
import pandas as pd

# Start from validated event + leakage-safe context
phase1 = event_context_final.copy()

# Remove merge-only column
if "timestamp" in phase1.columns:
    phase1 = phase1.drop(columns=["timestamp"])

# Reset row order
phase1 = phase1.reset_index(drop=True)
anomaly_complete = anomaly_complete.reset_index(drop=True)

# ------------------------------------------------------------
# SAFETY CHECK
# ------------------------------------------------------------

print("Event rows:", len(phase1))
print("Anomaly rows:", len(anomaly_complete))

assert len(phase1) == len(anomaly_complete)

# ------------------------------------------------------------
# Add stable row-based event ID
# ------------------------------------------------------------

phase1["event_id"] = np.arange(
    len(phase1),
    dtype="int64"
)

# ------------------------------------------------------------
# Add rebuilt anomaly results
# ------------------------------------------------------------

phase1["anomaly_prediction"] = (
    anomaly_complete["anomaly_prediction"].values
)

phase1["anomaly_score"] = (
    anomaly_complete["anomaly_score"].values
)

phase1["is_anomaly"] = (
    anomaly_complete["is_anomaly"].values
)

phase1["anomaly_method"] = (
    "Isolation Forest prototype"
)

# ------------------------------------------------------------
# FINAL CHECK
# ------------------------------------------------------------

print("\n========== PHASE 1 FINAL CHECK ==========")

print("Rows:", len(phase1))

print(
    "Unique events:",
    phase1["event_id"].nunique()
)

print(
    "Duplicate event IDs:",
    phase1["event_id"].duplicated().sum()
)

print("\nMissing values:")

missing = phase1.isna().sum()
missing = missing[missing > 0]

if len(missing) == 0:
    print("0 missing values")
else:
    print(missing)

print("\nAnomaly distribution:")

print(
    phase1["is_anomaly"].value_counts()
)

print("\nAnomaly rate:")

print(
    round(
        phase1["is_anomaly"].mean() * 100,
        2
    ),
    "%"
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

OUTPUT = (
    PROJECT_ROOT /
    "data/ml_ready/phase1_event_level_ml_dataset.csv"
)

OUTPUT.parent.mkdir(
    parents=True,
    exist_ok=True
)

phase1.to_csv(
    OUTPUT,
    index=False
)

print("\n============================================")
print("✅ PHASE 1 COMPLETE")
print("============================================")
print("\nSaved:")
print(OUTPUT)

Event rows: 439251
Anomaly rows: 439251

========== PHASE 1 FINAL CHECK ==========
Rows: 439251
Unique events: 439251
Duplicate event IDs: 0

Missing values:
0 missing values

Anomaly distribution:
is_anomaly
False    356761
True      82490
Name: count, dtype: int64

Anomaly rate:
18.78 %

✅ PHASE 1 COMPLETE

Saved:
C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\data\ml_ready\phase1_event_level_ml_dataset.csv
